# Notebook 02: Huấn luyện mô hình Seq2Seq trên Google Colab (GPU T4)
**Đề tài NLP-28:** Sửa lỗi ngữ pháp & chuẩn hóa phụ đề ASR tiếng Việt

Quy trình thực hiện:
1. Cấu hình môi trường Colab & kiểm tra GPU T4 (16GB VRAM).
2. Clone mã nguồn từ GitHub `lhhoang5425/vi-sub-corretion`.
3. Tạo tập dữ liệu huấn luyện Noisy -> Clean bằng `ASRNoiseGenerator`.
4. Fine-tuning mô hình nền `VietAI/vit5-base` bằng `Seq2SeqTrainer`.
5. Đánh giá kiểm thử (Evaluation) và nén mô hình để tải về máy.

## 1. Kiểm tra GPU và Clone dự án từ GitHub

In [ ]:
!nvidia-smi

In [ ]:
# Clone mã nguồn dự án
!git clone https://github.com/lhhoang5425/vi-sub-corretion.git
%cd vi-sub-corretion

# Cài đặt các thư viện cần thiết
!pip install -r requirements.txt -q

## 2. Chuẩn bị tập dữ liệu huấn luyện (Data Synthesis)
Sử dụng `ASRNoiseGenerator` để biến các câu tiếng Việt sạch thành cặp (noisy, clean).

In [ ]:
import json
from pathlib import Path
from src.data_processing.noise_generator import ASRNoiseGenerator

# Tập câu tiếng Việt mẫu đa dạng chủ đề (Bạn có thể nạp thêm từ file txt của bạn)
seed_sentences = [
    "Tôi đi làm ở ngân hàng và dành tiền tiết kiệm hàng tháng.",
    "Hôm nay thời tiết rất đẹp, chúng ta cùng đi dạo phố nhé.",
    "Bác sĩ khuyên bệnh nhân nên nghỉ ngơi và uống thuốc đúng giờ.",
    "Học phần xử lý ngôn ngữ tự nhiên giúp sinh viên hiểu sâu về trí tuệ nhân tạo.",
    "Phim này có phụ đề rất chuẩn xác và âm thanh sống động.",
    "Chúng tôi đang nghiên cứu mô hình ngôn ngữ lớn phục vụ cho doanh nghiệp.",
    "Khách hàng có thể tra cứu thông tin tài khoản qua ứng dụng di động.",
    "Để đạt kết quả học tập tốt, sinh viên cần chủ động làm bài tập lớn.",
    "Dự án này giúp chuẩn hóa phụ đề video một cách tự động và chính xác.",
    "Hệ thống phát hiện gian lận giao dịch ngân hàng hoạt động liên tục ngày đêm."
] * 500  # Nhân bản tạo 5,000 câu huấn luyện mẫu

generator = ASRNoiseGenerator()
train_data = []
val_data = []

for i, sent in enumerate(seed_sentences):
    noisy, errs = generator.apply_noise(sent)
    record = {"id": i + 1, "noisy": noisy, "clean": sent, "errors": errs}
    if i % 10 == 0:
        val_data.append(record)
    else:
        train_data.append(record)

Path("data/synthetic").mkdir(parents=True, exist_ok=True)
with open("data/synthetic/train.jsonl", "w", encoding="utf-8") as f:
    for r in train_data:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

with open("data/synthetic/val.jsonl", "w", encoding="utf-8") as f:
    for r in val_data:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Đã tạo xong: {len(train_data)} mẫu train và {len(val_data)} mẫu validation!")

## 3. Huấn luyện mô hình Seq2Seq (ViT5-base)

In [ ]:
# Khởi chạy huấn luyện bằng script đã chuẩn bị sẵn
!python -m src.models.train_seq2seq \
    --model_name "VietAI/vit5-base" \
    --train_file "data/synthetic/train.jsonl" \
    --val_file "data/synthetic/val.jsonl" \
    --output_dir "checkpoints/vit5-sub-corrector" \
    --epochs 3 \
    --batch_size 16 \
    --lr 5e-5

## 4. Kiểm thử mô hình vừa huấn luyện

In [ ]:
from src.models.inference import Seq2SeqCorrector

# Nạp mô hình vừa train từ checkpoints/
my_corrector = Seq2SeqCorrector(model_name_or_path="checkpoints/vit5-sub-corrector")

test_cases = [
    "toi di lam o ngan hang va gianh tien tiet kiem",
    "bac si bao phai nghi ngoi uong thuoc deu dan",
    "hom nay thoi tiet rat dep chung ta di choi nhe"
]

for text in test_cases:
    cleaned = my_corrector.predict(text)
    print(f"Input : {text}")
    print(f"Output: {cleaned}\n")

## 5. Nén checkpoint mô hình để tải về máy hoặc lưu vào Google Drive

In [ ]:
# Nén thư mục checkpoint
!zip -r vit5_checkpoint.zip checkpoints/vit5-sub-corrector

# Tải về máy tính của bạn
from google.colab import files
files.download("vit5_checkpoint.zip")

# Hoặc lưu vào Google Drive nếu muốn:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp vit5_checkpoint.zip /content/drive/MyDrive/